# Graphe de Synergies
**Objectif :** construire un graphe NetworkX où :
- Chaque **nœud** = une carte
- Chaque **arête** = une synergie mesurée (Jaccard >= 0.3, count >= 5)
- Le **poids** de l'arête = score Jaccard

Ce graphe permet d'identifier les cartes centrales d'un archetype,
les ponts entre archetypes, et l'impact d'un futur ban.

In [1]:
import sqlite3
import pandas as pd
import networkx as nx

con = sqlite3.connect('../data/yugioh.db')

# Charger les paires significatives
# Filtre : Jaccard >= 0.3 ET au moins 5 decks en commun
pairs = pd.read_sql("""
    SELECT card_a, card_b, jaccard, cooc_count
    FROM card_cooccurrence
    WHERE jaccard >= 0.3
    AND cooc_count >= 5
    ORDER BY jaccard DESC
""", con)

print(f'Paires chargées : {len(pairs):,}')

Paires chargées : 2,724


## 1. Construction du graphe

In [2]:
G = nx.Graph()

for _, row in pairs.iterrows():
    G.add_edge(row['card_a'], row['card_b'],
               weight=row['jaccard'],
               count=row['cooc_count'])

print(f'Noeuds (cartes) : {G.number_of_nodes():,}')
print(f'Aretes (synergies) : {G.number_of_edges():,}')
print(f'Composantes connexes : {nx.number_connected_components(G):,}')

Noeuds (cartes) : 663
Aretes (synergies) : 2,724
Composantes connexes : 58


## 2. Centralite — quelles cartes sont les plus connectees ?

In [3]:
degree = pd.Series(dict(G.degree(weight='weight')), name='weighted_degree')
degree = degree.sort_values(ascending=False)

print('Top 20 cartes les plus connectees :')
degree.head(20)

Top 20 cartes les plus connectees :


Lunalight Kaleido Chick          11.0657
Erebus the Underworld Monarch    10.7176
Eidos the Underworld Monarch     10.6686
Tenacity of the Monarchs         10.6429
Edea the Heavenly Squire         10.5212
Tessera the Primal Squire        10.4422
Pantheism of the Monarchs        10.4279
The Monarchs Masterplan          10.4279
The Monarchs Revolt              10.4279
Lunalight Silver Hound           10.3185
Lunalight Fusion                  9.6843
Lunalight Yellow Marten           9.5189
Luna Light Perfume                9.4770
Eidos the Underworld Squire       9.4620
Branded Fusion                    9.3159
Mementotlan Goblin                9.2656
Lunalight Black Sheep             9.2145
K9-66a Jokul                      9.1723
Lunalight Masquerade              9.0856
Once Upon a Fairy Tail            9.0833
Name: weighted_degree, dtype: float64

## 3. Communautes — detection automatique des archetypes

In [4]:
from networkx.algorithms.community import greedy_modularity_communities

communities = list(greedy_modularity_communities(G, weight='weight'))
print(f'{len(communities)} communautes detectees')
print()

communities_sorted = sorted(communities, key=len, reverse=True)
for i, comm in enumerate(communities_sorted[:8]):
    cards_list = sorted(comm)[:6]
    print(f'Communaute {i+1} ({len(comm)} cartes) : {", ".join(cards_list)}...')

63 communautes detectees

Communaute 1 (43 cartes) : Albion the Shrouded Dragon, Aluber the Jester of Despia, Blazing Cartesia, the Virtuous, Branded Fusion, Branded Lost, Branded Opening...
Communaute 2 (26 cartes) : Forbidden Crown, Forbidden Droplet, Mystical Space Typhoon, Noble Knight's Shield-Bearer, Pot of Desires, Radiant Typhoon Ascendance...
Communaute 3 (25 cartes) : Ame no Habakiri no Mitsurugi, Ame no Murakumo no Mitsurugi, Bonfire, End of the World Ruler, Ext Ryzeal, Futsu no Mitama no Mitsurugi...
Communaute 4 (22 cartes) : Astellar of the White Forest, Azamina Debtors, Curse of Diabell, Deception of the Sinful Spoils, Diabellstar Vengeance, Diabellstar the Black Witch...
Communaute 5 (21 cartes) : Cooky☆Yummy, Cupsy☆Yummy, Dark Magical Curtain, Dark Magician, Dark Magician, the Pharaoh's Servant, Fabled Lurrie...
Communaute 6 (19 cartes) : Allure of Darkness, Backup @Ignister, Bystial Baldrake, Bystial Druiswurm, Bystial Magnamhut, Code of Soul...
Communaute 7 (19 carte

In [5]:
## Classification des cartes — format staples vs archetype pieces

import sqlite3, pandas as pd

con_tmp = sqlite3.connect('../data/yugioh.db')

# ── Fréquence globale (% decks légaux contenant la carte, main deck) ──────────
total_decks = pd.read_sql(
    "SELECT COUNT(*) as n FROM tournament_decks WHERE illegal = 0", con_tmp
).iloc[0]['n']

freq_df = pd.read_sql("""
    SELECT dc.card_name,
           COUNT(DISTINCT dc.deck_id) AS deck_count
    FROM deck_cards dc
    JOIN tournament_decks td ON td.id = dc.deck_id
    WHERE td.illegal = 0 AND dc.zone = 'main'
    GROUP BY dc.card_name
""", con_tmp)
freq_df['frequency'] = freq_df['deck_count'] / total_decks

# ── Dispersion cross-archetype : dans combien d'archetypes distincts ──────────
# (calculé sur les données brutes, pas le graphe)
arch_df = pd.read_sql("""
    SELECT dc.card_name, td.archetype
    FROM deck_cards dc
    JOIN tournament_decks td ON td.id = dc.deck_id
    WHERE td.illegal = 0 AND dc.zone = 'main' AND td.archetype IS NOT NULL
""", con_tmp)
con_tmp.close()

arch_spread = (arch_df.groupby('card_name')['archetype']
               .nunique()
               .reset_index()
               .rename(columns={'archetype': 'n_archetypes'}))

# ── Fusion & classification ───────────────────────────────────────────────────
card_class = freq_df.merge(arch_spread, on='card_name', how='left')
card_class['n_archetypes'] = card_class['n_archetypes'].fillna(1).astype(int)
card_class['frequency_pct'] = (card_class['frequency'] * 100).round(1)

# Seuils :
#   frequency > 25%  → carte jouée dans beaucoup de decks
#   n_archetypes > 5 → carte présente dans plusieurs archetypes distincts
def classify(row):
    hi_freq  = row['frequency'] > 0.25
    hi_cross = row['n_archetypes'] > 5
    if hi_freq and hi_cross:     return 'staple_format'
    if hi_freq and not hi_cross: return 'staple_archetype'
    if not hi_freq and hi_cross: return 'tech_pont'
    return 'piece_niche'

card_class['card_type'] = card_class.apply(classify, axis=1)

print('Distribution des types :')
print(card_class['card_type'].value_counts().to_string())
print()

print('=== STAPLES DE FORMAT (présentes dans beaucoup de decks ET archetypes) ===')
sf = card_class[card_class['card_type'] == 'staple_format'].sort_values('frequency', ascending=False)
print(sf[['card_name', 'frequency_pct', 'n_archetypes']].head(25).to_string(index=False))
print()

print('=== CARTES PONT (tech présente dans plusieurs archetypes, peu fréquente) ===')
tp = card_class[card_class['card_type'] == 'tech_pont'].sort_values('n_archetypes', ascending=False)
print(tp[['card_name', 'frequency_pct', 'n_archetypes']].head(20).to_string(index=False))
print()

print('=== STAPLES ARCHETYPE (dominant dans un seul archetype fort) ===')
sa = card_class[card_class['card_type'] == 'staple_archetype'].sort_values('frequency', ascending=False)
print(sa[['card_name', 'frequency_pct', 'n_archetypes']].head(20).to_string(index=False))

Distribution des types :
card_type
piece_niche      1842
tech_pont         211
staple_format      11

=== STAPLES DE FORMAT (présentes dans beaucoup de decks ET archetypes) ===
                    card_name  frequency_pct  n_archetypes
  Ash Blossom & Joyous Spring           82.9           117
            Mulcharmy Fuwalos           65.7           111
          Called by the Grave           58.6           116
                     Maxx "C"           43.3            78
            Droll & Lock Bird           42.3            90
Ghost Belle & Haunted Mansion           40.0            82
        Infinite Impermanence           39.8            98
    The Fallen & The Virtuous           31.7            54
                Effect Veiler           27.6            58
            Forbidden Droplet           27.3            90
        Triple Tactics Talent           27.3            88

=== CARTES PONT (tech présente dans plusieurs archetypes, peu fréquente) ===
                    card_name  freque

## 4. Focus sur un archetype — sous-graphe Maliss

In [6]:
def archetype_subgraph(G, keyword, min_jaccard=0.3):
    nodes = [n for n in G.nodes() if keyword.lower() in n.lower()]
    neighbors = set(nodes)
    for n in nodes:
        for neighbor, data in G[n].items():
            if data['weight'] >= min_jaccard:
                neighbors.add(neighbor)
    return G.subgraph(neighbors)

maliss_sg = archetype_subgraph(G, 'Maliss')
print(f'Sous-graphe Maliss : {maliss_sg.number_of_nodes()} cartes, {maliss_sg.number_of_edges()} synergies')
print()

maliss_degree = pd.Series(dict(maliss_sg.degree(weight='weight'))).sort_values(ascending=False)
print('Cartes les plus centrales dans Maliss :')
maliss_degree.head(10)

Sous-graphe Maliss : 16 cartes, 88 synergies

Cartes les plus centrales dans Maliss :


Maliss C MTP-07          8.3784
Maliss P Dormouse        8.3784
Wizard @Ignister         8.2458
Maliss in Underground    7.1758
Maliss P White Rabbit    7.1619
Maliss P March Hare      7.1004
Maliss P Chessy Cat      6.6346
Maliss C TB-11           6.6007
Allure of Darkness       6.5490
Backup @Ignister         6.3568
dtype: float64

## 5. Simulation de ban — impact sur le graphe

In [7]:
def simulate_ban(G, card_name):
    if card_name not in G:
        print(f'Carte "{card_name}" introuvable dans le graphe.')
        return
    
    neighbors = list(G.neighbors(card_name))
    weights = [G[card_name][n]['weight'] for n in neighbors]
    impact = pd.Series(weights, index=neighbors).sort_values(ascending=False)
    
    print(f'Ban de "{card_name}"')
    print(f'  Connexions supprimees : {len(neighbors)}')
    print(f'  Cartes les plus impactees :')
    for card, w in impact.head(10).items():
        print(f'    {w:.2f}  {card}')

simulate_ban(G, 'Ash Blossom & Joyous Spring')

Ban de "Ash Blossom & Joyous Spring"
  Connexions supprimees : 6
  Cartes les plus impactees :
    0.64  Mulcharmy Fuwalos
    0.48  Ghost Belle & Haunted Mansion
    0.44  Infinite Impermanence
    0.33  Effect Veiler
    0.32  Droll & Lock Bird
    0.31  Fydraulis Harmonia


In [8]:
simulate_ban(G, 'Maliss P March Hare')

Ban de "Maliss P March Hare"
  Connexions supprimees : 13
  Cartes les plus impactees :
    0.87  Allure of Darkness
    0.79  Maliss P Chessy Cat
    0.71  Backup @Ignister
    0.68  Maliss in Underground
    0.68  Maliss P White Rabbit
    0.52  Maliss P Dormouse
    0.52  Maliss C MTP-07
    0.52  Wizard @Ignister
    0.40  Bystial Druiswurm
    0.39  Maliss C TB-11


## 6. Visualisation interactive — Pyvis

In [9]:
from pyvis.network import Network

def visualize_archetype(G, keyword, min_jaccard=0.3, output_file=None):
    """
    Génère un graphe interactif HTML pour un archetype.
    - Taille du noeud = centralité
    - Épaisseur de l'arête = score Jaccard
    - Rouge = cartes core de l'archetype, Bleu = cartes externes
    """
    sg = archetype_subgraph(G, keyword, min_jaccard)
    core_cards = set(n for n in sg.nodes() if keyword.lower() in n.lower())

    net = Network(height='700px', width='100%', bgcolor='#1a1a2e', font_color='white')
    net.set_options('''
    {
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -80,
          "springLength": 120
        },
        "solver": "forceAtlas2Based"
      }
    }
    ''')

    degrees = dict(sg.degree(weight='weight'))
    max_deg = max(degrees.values()) if degrees else 1

    for node in sg.nodes():
        size = 15 + 35 * (degrees[node] / max_deg)
        color = '#e94560' if node in core_cards else '#4a90d9'
        net.add_node(node, label=node, size=size, color=color,
                     title=f'Centralité: {degrees[node]:.2f}')

    for u, v, data in sg.edges(data=True):
        net.add_edge(u, v, value=data['weight'],
                     title=f"Jaccard: {data['weight']:.2f} ({data['count']} decks)")

    if output_file is None:
        output_file = f'../data/graph_{keyword.lower().replace(" ", "_")}.html'

    net.save_graph(output_file)
    print(f'✓ {output_file} — {sg.number_of_nodes()} cartes, {sg.number_of_edges()} synergies')
    return output_file

# Générer les graphes des top archetypes
for archetype in ['Maliss', 'Tenpai', 'Ryzeal', 'Branded']:
    visualize_archetype(G, archetype)


✓ ../data/graph_maliss.html — 16 cartes, 88 synergies
✓ ../data/graph_tenpai.html — 6 cartes, 15 synergies
✓ ../data/graph_ryzeal.html — 18 cartes, 74 synergies
✓ ../data/graph_branded.html — 40 cartes, 201 synergies


## 7. Betweenness Centrality — cartes pont entre archetypes (TOK-16)

## 7. Betweenness Centrality — cartes pont entre archetypes (TOK-16)

In [10]:
# Betweenness centrality : cartes pont entre archetypes (TOK-16)
# Approximée sur k=500 sources pour performance (exact = O(VE) trop lent sur graphe complet)

import pandas as pd

bc = nx.betweenness_centrality(G, weight='weight', normalized=True, k=min(500, G.number_of_nodes()))
dc = nx.degree_centrality(G)
cc = nx.closeness_centrality(G, distance='weight')

metrics = pd.DataFrame({
    'card_name':         list(bc.keys()),
    'betweenness':       [round(bc[n], 6) for n in bc],
    'degree_centrality': [round(dc[n], 6) for n in bc],
    'closeness':         [round(cc.get(n, 0), 6) for n in bc],
    'degree_weighted':   [round(G.degree(n, weight='weight'), 4) for n in bc],
})
metrics = metrics.sort_values('betweenness', ascending=False)

print('Top 20 cartes par betweenness centrality (ponts entre archetypes) :')
print(metrics.head(20).to_string(index=False))

# Sauvegarder
con_bc = sqlite3.connect('../data/yugioh.db')
con_bc.execute('DROP TABLE IF EXISTS card_graph_metrics')
con_bc.execute('''
    CREATE TABLE card_graph_metrics (
        card_name         TEXT PRIMARY KEY,
        betweenness       REAL,
        degree_centrality REAL,
        closeness         REAL,
        degree_weighted   REAL
    )
''')
metrics.to_sql('card_graph_metrics', con_bc, if_exists='append', index=False)
con_bc.commit()
con_bc.close()
print(f'\n✓ {len(metrics)} cartes sauvegardées dans card_graph_metrics')

Top 20 cartes par betweenness centrality (ponts entre archetypes) :
                                  card_name  betweenness  degree_centrality  closeness  degree_weighted
                 Albion the Shrouded Dragon     0.005754           0.022659   0.105649           6.5106
                      Triple Tactics Thrust     0.005488           0.003021   0.091835           0.6273
                      Triple Tactics Talent     0.005272           0.003021   0.081176           0.6390
                        Called by the Grave     0.005045           0.004532   0.071777           1.3223
                           Gold Sarcophagus     0.004432           0.031722   0.100302           7.6626
                             Springans Kitt     0.004033           0.016616   0.103962           4.3782
                           Branded in White     0.002983           0.010574   0.087021           2.7667
                                   Maxx "C"     0.002471           0.004532   0.058522           1.3

## 8. Nommage automatique des communautés (TOK-17)

In [11]:
# Nommage automatique des communautés du graphe (TOK-17)
# Nom = carte avec le degré pondéré le plus élevé dans la communauté
# Label étendu = archetype depuis la DB

con_nm = sqlite3.connect('../data/yugioh.db')
arch_map = pd.read_sql('SELECT name, archetype FROM cards WHERE archetype IS NOT NULL', con_nm)
arch_map = dict(zip(arch_map['name'], arch_map['archetype']))

community_rows = []
for i, comm in enumerate(communities_sorted):
    # Carte la plus centrale (degré pondéré)
    comm_list = list(comm)
    if not comm_list:
        continue
    
    subg = G.subgraph(comm_list)
    local_deg = dict(subg.degree(weight='weight'))
    lead_card  = max(local_deg, key=local_deg.get)
    lead_arch  = arch_map.get(lead_card, lead_card)
    
    # Top 3 archetypes représentés dans la communauté
    arches = [arch_map.get(c, c) for c in comm_list if arch_map.get(c)]
    from collections import Counter
    top_arches = [a for a, _ in Counter(arches).most_common(3)]
    
    community_rows.append({
        'community_id':   i,
        'n_cards':        len(comm_list),
        'lead_card':      lead_card,
        'archetype_label': lead_arch,
        'top_archetypes': ', '.join(top_arches),
        'lead_degree':    round(local_deg[lead_card], 3),
    })

comm_df = pd.DataFrame(community_rows)

print(f'{len(comm_df)} communautés nommées :')
print(comm_df[['community_id','n_cards','lead_card','archetype_label','top_archetypes']]
      .to_string(index=False))

# Sauvegarder
con_nm.execute('DROP TABLE IF EXISTS graph_communities')
con_nm.execute('''
    CREATE TABLE graph_communities (
        community_id    INTEGER PRIMARY KEY,
        n_cards         INTEGER,
        lead_card       TEXT,
        archetype_label TEXT,
        top_archetypes  TEXT,
        lead_degree     REAL
    )
''')
comm_df.to_sql('graph_communities', con_nm, if_exists='append', index=False)
con_nm.commit()
con_nm.close()
print(f'\n✓ {len(comm_df)} communautés sauvegardées dans graph_communities')

63 communautés nommées :
 community_id  n_cards                             lead_card                      archetype_label                            top_archetypes
            0       43                        Branded Fusion                              Branded                Dracotail, Branded, Despia
            1       26               Radiant Typhoon Meghala                      Radiant Typhoon   Radiant Typhoon, Sky Striker, Forbidden
            2       25             Mitsurugi no Mikoto, Saji                            Mitsurugi                Mitsurugi, Ryzeal, Seventh
            3       22             Tales of the White Forest                         White Forest      White Forest, Sinful Spoils, Diabell
            4       21                        Yummy☆Surprise                                Yummy          Yummy, Dark Magician, Fiendsmith
            5       19                       Maliss C MTP-07                      Maliss C MTP-07                Maliss, @Ignister, Bys


✓ 63 communautés sauvegardées dans graph_communities
